# DDPG & TD3 -- Controle continu off-policy sur HalfCheetah

**Algorithmes** : DDPG (Deep Deterministic Policy Gradient) et TD3 (Twin Delayed DDPG)
**Environnement** : HalfCheetah-v5 (MuJoCo, Gymnasium)
**Papiers** : [Silver et al., 2014](https://proceedings.mlr.press/v32/silver14.html) (DPG) ; [Lillicrap et al., 2015](https://arxiv.org/abs/1509.02971) (DDPG) ; [Fujimoto et al., 2018](https://arxiv.org/abs/1802.09477) (TD3)

> **Quick Facts** (inspires de [SpinningUp](https://spinningup.openai.com)) :
> - DDPG et TD3 sont des algorithmes **off-policy**.
> - Ils ne fonctionnent que pour les espaces d'action **continus**.
> - DDPG peut etre vu comme du **deep Q-learning adapte au continu** : au lieu
>   de chercher $\arg\max_a Q(s,a)$ par enumeration (impossible en continu),
>   on apprend une politique $\mu_\theta(s)$ qui approxime ce max via le gradient.
> - TD3 corrige les trois faiblesses de DDPG avec trois mecanismes simples.

| Propriete | DDPG | TD3 |
|-----------|------|-----|
| Model-free | oui | oui |
| Off-policy | oui | oui |
| Politique deterministe | oui | oui |
| Replay buffer | oui | oui |
| Twin critics | non | oui |
| Delayed policy updates | non | oui |
| Target policy smoothing | non | oui |
| Biais d'overestimation | fort | attenue |
| Stabilite empirique | moderee | bonne |

Jusqu'ici, on a vu plusieurs familles d'algorithmes de RL :

- **Q-learning / SARSA** : methodes value-based tabulaires.
- **DQN / Double DQN** : value-based avec reseau de neurones, replay buffer, target network.
- **REINFORCE / A2C / A2C-GAE** : policy-gradient / actor-critic on-policy.
- **TRPO / PPO** : actor-critic on-policy stabilise par trust region ou clipping.

DDPG et TD3 se placent au croisement de ces mondes : **off-policy** comme DQN (replay buffer, target networks), **actor-critic** comme A2C (politique parametree + critique), mais avec une politique **deterministe** et un critique $Q(s,a)$ qui prend l'action en entree.

## HalfCheetah-v5 : l'environnement

HalfCheetah est un robot bipede simplifie dans un plan 2D. L'objectif est de courir le plus vite possible vers la droite. Contrairement a CartPole (discret, 1D), HalfCheetah a un espace d'action **continu et multidimensionnel** -- ce qui rend DQN inutilisable directement.

### Espace d'observation -- 17 dimensions

| Groupe | Dimensions | Description |
|--------|-----------|-------------|
| Positions articulaires | 0-5 | Angles des 6 articulations |
| Vitesses articulaires | 6-11 | Vitesses angulaires des 6 articulations |
| Vitesse du corps | 12-14 | Vitesse lineaire du centre de masse |
| Vitesse angulaire | 15-16 | Vitesse angulaire du corps |

### Espace d'actions -- 6 dimensions continues

| Dimension | Articulation | Plage |
|-----------|-------------|-------|
| 0 | Hanche avant | $[-1, 1]$ |
| 1 | Genou avant | $[-1, 1]$ |
| 2 | Cheville avant | $[-1, 1]$ |
| 3 | Hanche arriere | $[-1, 1]$ |
| 4 | Genou arriere | $[-1, 1]$ |
| 5 | Cheville arriere | $[-1, 1]$ |

### Structure de la recompense

$$r_t = \underbrace{v_x}_{\text{vitesse vers la droite}} - \underbrace{0.1 \|a_t\|^2}_{\text{penalite d'effort}}$$

Il n'y a pas de condition de fin d'episode autre que la limite de temps (1000 pas).

**Difference cle avec CartPole.** CartPole a 2 actions discretes -- DQN peut enumerer toutes les valeurs Q. HalfCheetah a $[-1,1]^6$ continu -- impossible d'enumerer. C'est pourquoi on a besoin d'un **reseau acteur** qui propose directement l'action optimale.

In [ ]:
import copy
import time
from collections import deque
from pathlib import Path

import gymnasium as gym
from gymnasium.wrappers import RecordVideo
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

try:
    from IPython.display import Video, display
except Exception:  # pragma: no cover - optionnel dans un export script
    Video = display = None

try:
    from tqdm.auto import tqdm
except Exception:  # pragma: no cover - notebook minimal sans tqdm
    tqdm = None

try:
    from scipy import stats
    from scipy.ndimage import uniform_filter1d
except Exception:  # pragma: no cover - visualisations optionnelles
    stats = None
    uniform_filter1d = None

%matplotlib inline
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

print(f"PyTorch : {torch.__version__}")
print(f"Gymnasium : {gym.__version__}")

try:
    env_test = gym.make("HalfCheetah-v5")
    env_test.close()
    print("MuJoCo : disponible")
except Exception as e:
    print(f"MuJoCo : ERREUR -> {e}")

In [ ]:
env_probe = gym.make("HalfCheetah-v5")
OBS_DIM = env_probe.observation_space.shape[0]
ACTION_DIM = env_probe.action_space.shape[0]
ACTION_LOW = env_probe.action_space.low.astype(np.float32)
ACTION_HIGH = env_probe.action_space.high.astype(np.float32)

print("Observation space :", env_probe.observation_space)
print("Action space      :", env_probe.action_space)
print("OBS_DIM           :", OBS_DIM)
print("ACTION_DIM        :", ACTION_DIM)
print("Action low/high   :", ACTION_LOW, ACTION_HIGH)

env_probe.close()


## 1. Du gradient de politique stochastique au gradient deterministe

### Rappel : gradient stochastique (PPO / A2C)

Dans PPO et A2C, la politique $\pi_\theta(a|s)$ est stochastique -- elle produit une distribution sur les actions. Le theoreme du gradient de politique donne :

$$\nabla_\theta J(\theta) = \mathbb{E}_{s,\, a \sim \pi_\theta} \left[ \nabla_\theta \log \pi_\theta(a|s) \cdot Q^\pi(s,a) \right]$$

En grande dimension, cet estimateur est tres bruite : il faut echantillonner des actions pour estimer l'esperance.

### Le theoreme DPG (Silver et al., 2014)

Plutot qu'une distribution, on parametre directement une **fonction** $\mu_\theta : \mathcal{S} \to \mathcal{A}$ qui mappe chaque etat a une action unique. Le gradient de politique **deterministe** (DPG) est :

$$\boxed{\nabla_\theta J(\theta) = \mathbb{E}_{s \sim \rho^\mu} \left[ \nabla_a Q^\mu(s,a) \big|_{a=\mu_\theta(s)} \cdot \nabla_\theta \mu_\theta(s) \right]}$$

**Lecture.** On evalue la pente de $Q$ par rapport a l'action en $a = \mu_\theta(s)$, puis on la propage a travers le Jacobien de la politique. Par la regle de la chaine : "si bouger l'action vers la droite augmente $Q$, dans quelle direction bouger $\theta$ ?"

**Analogie.** La politique stochastique est un archer qui tire des fleches aleatoires et observe les impacts. La politique deterministe est un archer qui pose sa fleche sur la cible et la deplace selon la pente -- beaucoup plus efficace, mais seulement si la cible (le critique $Q$) est bien calibree.

| | Stochastique (PPO) | Deterministe (DDPG) |
|-|--------------------|--------------------|
| Gradient | $\mathbb{E}_{a \sim \pi}[\nabla_\theta \log \pi \cdot Q]$ | $\mathbb{E}_s[\nabla_a Q \cdot \nabla_\theta \mu_\theta]$ |
| Integrale sur les actions | oui (echantillonnage) | non (gradient direct) |
| Variance du gradient | elevee | basse |
| Exploration | intrinseque (distribution) | externe (bruit ajoute) |

Nous avons maintenant le theoreme. Visualisons comment le gradient deterministe opere sur une surface Q synthetique.

In [ ]:
# === Viz DPG : le gradient deterministe sur une surface Q synthetique ===

fig, ax = plt.subplots(figsize=(8, 5))

a_range = np.linspace(-1.5, 1.5, 300)
a_optimal = 0.7
np.random.seed(0)

Q_true = -(a_range - a_optimal)**2 + 1.0
Q_estimated = Q_true + 0.15 * np.sin(5 * a_range) + 0.05 * np.random.randn(len(a_range))

mu_current = 0.1
Q_at_mu = np.interp(mu_current, a_range, Q_estimated)
da = a_range[1] - a_range[0]
idx_mu = np.argmin(np.abs(a_range - mu_current))
dQ_da = (Q_estimated[idx_mu + 1] - Q_estimated[idx_mu - 1]) / (2 * da)

ax.plot(a_range, Q_true, 'b-', lw=2, label='$Q^{vrai}(s, a)$', alpha=0.5)
ax.plot(a_range, Q_estimated, 'r-', lw=1.5, label='$Q_\\phi(s, a)$ estime', alpha=0.8)
ax.axvline(a_optimal, color='b', linestyle='--', alpha=0.5, label=f'$a^*={a_optimal}$ (optimal)')
ax.scatter([mu_current], [Q_at_mu], color='green', s=100, zorder=5,
           label=f'$\\mu_\\theta(s)={mu_current}$ (actuel)')

arrow_scale = 0.3
ax.annotate('', xy=(mu_current + arrow_scale * np.sign(dQ_da), Q_at_mu + 0.05),
            xytext=(mu_current, Q_at_mu),
            arrowprops=dict(arrowstyle='->', color='green', lw=2.5))
ax.text(mu_current + arrow_scale * np.sign(dQ_da) + 0.05, Q_at_mu + 0.1,
        '$\\nabla_a Q_\\phi$', color='green', fontsize=13)

ax.set_xlabel('Action $a$', fontsize=12)
ax.set_ylabel('$Q(s, a)$', fontsize=12)
ax.set_title("Gradient DPG : l'acteur suit $\\nabla_a Q$ sur la surface du critique", fontsize=13)
ax.legend(fontsize=10)
ax.set_xlim(-1.5, 1.5)
plt.tight_layout()
plt.show()

**Interpretation.** La fleche verte montre le gradient $\nabla_a Q_\phi$ evalue en $\mu_\theta(s) = 0.1$. La pente est positive : augmenter l'action ameliore $Q$. L'acteur est donc mis a jour pour deplacer $\mu_\theta(s)$ vers la droite, en direction de $a^* = 0.7$.

Ce plot revele aussi une limite : l'acteur suit la pente du **critique estime** $Q_\phi$, pas celle du vrai $Q$. Si $Q_\phi$ a des pics artificiels, l'acteur peut converger vers une action incorrecte. C'est l'une des raisons pour lesquelles DDPG peut etre instable -- et pourquoi TD3 ajoute des corrections.

Maintenant que le theoreme DPG est clair, construisons les deux reseaux de DDPG : l'acteur et le critique.

## 2. L'acteur deterministe $\mu_\theta(s)$

L'acteur est un MLP qui mappe une observation a une action continue :

$$\mu_\theta : \mathbb{R}^{obs\_dim} \to \mathbb{R}^{action\_dim}$$

La couche de sortie utilise $\tanh$ pour borner les actions dans $[-1, 1]$, puis une transformation affine pour rescaler dans $[a_{\min}, a_{\max}]$ :

$$\mu_\theta(s) = \underbrace{\frac{a_{\max} - a_{\min}}{2}}_{\text{scale}} \cdot \tanh(W_{out} h + b_{out}) + \underbrace{\frac{a_{\max} + a_{\min}}{2}}_{\text{bias}}$$

Pour HalfCheetah-v5, $a \in [-1,1]^6$, donc scale = 1 et bias = 0. Le $\tanh$ garantit que l'acteur ne peut jamais produire d'action hors des bornes physiques de l'environment.

In [ ]:
class DeterministicActor(nn.Module):
    """Politique deterministe mu_theta(s) pour actions continues bornees.

    MLP(obs_dim -> 256 ReLU -> 256 ReLU -> action_dim tanh rescale)
    """

    def __init__(self, obs_dim, action_dim, hidden_dim=256,
                 action_low=-1.0, action_high=1.0):
        super().__init__()
        self.fc1 = nn.Linear(obs_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.mu = nn.Linear(hidden_dim, action_dim)

        self.register_buffer(
            "action_scale",
            torch.as_tensor((action_high - action_low) / 2.0, dtype=torch.float32),
        )
        self.register_buffer(
            "action_bias",
            torch.as_tensor((action_high + action_low) / 2.0, dtype=torch.float32),
        )
        self._init_weights()

    def _init_weights(self):
        for layer in [self.fc1, self.fc2]:
            nn.init.orthogonal_(layer.weight, gain=np.sqrt(2))
            nn.init.zeros_(layer.bias)
        nn.init.orthogonal_(self.mu.weight, gain=0.01)
        nn.init.zeros_(self.mu.bias)

    def forward(self, state):
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        return torch.tanh(self.mu(x)) * self.action_scale + self.action_bias

In [ ]:
# Mini-test acteur : verifier formes et bornes

actor = DeterministicActor(OBS_DIM, ACTION_DIM, action_low=ACTION_LOW, action_high=ACTION_HIGH)

obs_fake = torch.zeros(1, OBS_DIM)
action_out = actor(obs_fake)
print(f"Entree  : obs shape = {obs_fake.shape}")
print(f"Sortie  : action shape = {action_out.shape}")
print(f"Valeurs : {action_out.detach().numpy().round(4)}")
print(f"Min/Max : {action_out.min().item():.4f} / {action_out.max().item():.4f}")
print(f"Bornes attendues : [{ACTION_LOW[0]}, {ACTION_HIGH[0]}]")

# Batch de 4 observations
batch = torch.randn(4, OBS_DIM)
batch_out = actor(batch)
print(f"\nBatch de 4 : sortie shape = {batch_out.shape}")
print("Test acteur : OK")

**Ce qu'on observe.** L'acteur initialise avec gain=0.01 pour la couche de sortie produit des actions tres proches de zero. C'est intentionnel : au debut de l'entrainement, la politique est presque neutre, ce qui laisse la place au bruit d'exploration d'agir librement. Les actions restent dans $[-1, 1]$ grace au tanh.

Passons au critique, qui joue un role symetrique mais different : au lieu de produire une action, il juge une action donnee.

## 3. Le critique $Q_\phi(s, a)$

Le critique est un MLP qui prend en entree la **concatenation** de l'observation et de l'action :

$$Q_\phi : \mathbb{R}^{obs\_dim + action\_dim} \to \mathbb{R}$$

Pourquoi concatener $[s; a]$ en entree ? Cela permet au critique d'apprendre des interactions entre etat et action des la premiere couche.

**Comparaison des familles d'algorithmes :**

| Famille | Entree du critique/reseau Q | Sortie |
|---------|----------------------------|--------|
| DQN | $s$ | une valeur par action discrete |
| A2C / PPO | $s$ | $V(s)$, valeur moyenne de l'etat |
| DDPG / TD3 | $(s, a)$ | $Q(s,a)$, valeur d'une action continue precise |

Dans DQN, on peut faire $\arg\max_a Q(s,a)$ par enumeration car l'espace est discret. Ici, l'espace est continu : on ne peut pas enumerer. Le reseau acteur joue le role de l'argmax -- il propose directement l'action qui maximise $Q$.

In [ ]:
class ContinuousQNetwork(nn.Module):
    """Reseau Q(s, a; phi) -- concatene [s; a] en entree, produit un scalaire.

    MLP(obs_dim + action_dim -> 256 ReLU -> 256 ReLU -> 1)
    """

    def __init__(self, obs_dim, action_dim, hidden_dim=256):
        super().__init__()
        self.fc1 = nn.Linear(obs_dim + action_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.q_out = nn.Linear(hidden_dim, 1)
        self._init_weights()

    def _init_weights(self):
        for layer in [self.fc1, self.fc2]:
            nn.init.orthogonal_(layer.weight, gain=np.sqrt(2))
            nn.init.zeros_(layer.bias)
        nn.init.orthogonal_(self.q_out.weight, gain=1.0)
        nn.init.zeros_(self.q_out.bias)

    def forward(self, state, action):
        x = torch.cat([state, action], dim=-1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.q_out(x).squeeze(-1)  # (batch,)

In [ ]:
# Mini-test critique : verifier formes et valeurs

critic = ContinuousQNetwork(OBS_DIM, ACTION_DIM)

obs_fake = torch.zeros(1, OBS_DIM)
act_fake = torch.zeros(1, ACTION_DIM)
q_val = critic(obs_fake, act_fake)
print(f"Entree  : obs={obs_fake.shape}, action={act_fake.shape}")
print(f"Sortie  : Q shape = {q_val.shape}, valeur = {q_val.item():.4f}")

# Batch de 4
obs_b = torch.randn(4, OBS_DIM)
act_b = torch.randn(4, ACTION_DIM).clamp(-1, 1)
q_b = critic(obs_b, act_b)
print(f"\nBatch de 4 : Q shape = {q_b.shape}")
print(f"Valeurs : {q_b.detach().numpy().round(4)}")
print("Test critique : OK")

**Ce qu'on observe.** Le critique initialise produit des valeurs Q proches de zero. En debut d'entrainement, ces estimations sont sans signification -- le critique apprend progressivement a distinguer les bonnes actions des mauvaises.

Acteur et critique sont construits. Pour les entrainer en off-policy, il nous faut un buffer de replay et des reseaux cibles stables.

## 4. Off-policy et buffer de replay

### Pourquoi off-policy ?

Une politique deterministe $\mu_\theta$ mappe chaque etat a exactement une action. Sans variabilite, l'agent ne decouvre jamais de nouvelles regions de l'espace d'action. La solution : **ajouter du bruit externe** pendant la collecte.

$$a_t = \mu_\theta(s_t) + \varepsilon_t$$

Ce bruit signifie que les donnees collectees ne viennent pas de $\mu_\theta$ mais d'une politique "bruitee". L'apprentissage est donc **off-policy**. L'equation de Bellman reste valide hors politique :

$$y = r + \gamma (1 - d) \cdot Q_{\phi'}(s', \mu_{\theta'}(s'))$$

Cette cible ne depend que de la transition $(s, a, r, s')$ -- peu importe quelle politique l'a generee. C'est ce qui permet de **reutiliser les transitions passees** via un buffer.

### Le buffer de replay

Le buffer stocke les transitions $(s, a, r, s', \text{done})$ et les reechantillonne aleatoirement. Deux effets :

1. **Decorrelation temporelle** : les mises a jour successives utilisent des transitions melangees, brisant les correlations qui destabilisent le gradient.
2. **Reutilisation des donnees** : chaque transition peut etre utilisee des dizaines de fois -- bien meilleur que les methodes on-policy qui jettent les donnees apres utilisation.

In [ ]:
class ContinuousReplayBuffer:
    """Buffer de replay FIFO pour actions continues.

    Stocke (s, a, r, s', done) ou a est un vecteur float32.
    Retourne des tenseurs PyTorch pour le calcul de gradient.
    """

    def __init__(self, capacity):
        self._buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        """Ajoute une transition au buffer."""
        self._buffer.append((
            np.asarray(state, dtype=np.float32),
            np.asarray(action, dtype=np.float32),
            float(reward),
            np.asarray(next_state, dtype=np.float32),
            bool(done),
        ))

    def sample(self, batch_size):
        """Echantillonne un batch aleatoire sans remise."""
        indices = np.random.choice(len(self._buffer), size=batch_size, replace=False)
        states, actions, rewards, next_states, dones = zip(
            *(self._buffer[i] for i in indices)
        )
        return (
            torch.tensor(np.array(states), dtype=torch.float32),
            torch.tensor(np.array(actions), dtype=torch.float32),
            torch.tensor(np.array(rewards), dtype=torch.float32),
            torch.tensor(np.array(next_states), dtype=torch.float32),
            torch.tensor(np.array(dones), dtype=torch.float32),
        )

    def __len__(self):
        return len(self._buffer)

In [ ]:
# Mini-test buffer : push, sample, verifier formes

buf = ContinuousReplayBuffer(capacity=1000)
print(f"Buffer vide : len = {len(buf)}")

for _ in range(50):
    s = np.random.randn(OBS_DIM).astype(np.float32)
    a = np.random.uniform(-1, 1, ACTION_DIM).astype(np.float32)
    r = float(np.random.randn())
    s2 = np.random.randn(OBS_DIM).astype(np.float32)
    buf.push(s, a, r, s2, False)

print(f"Apres 50 push : len = {len(buf)}")

states, actions, rewards, next_states, dones = buf.sample(batch_size=8)
print(f"\nBatch de 8 :")
print(f"  states      : {states.shape}")
print(f"  actions     : {actions.shape}")
print(f"  rewards     : {rewards.shape}")
print(f"  next_states : {next_states.shape}")
print(f"  dones       : {dones.shape}")
print("Test buffer : OK")

### Note : `terminated` vs `truncated`

Gymnasium separe deux facons de finir un episode :

- `terminated=True` : etat terminal MDP reel. On coupe le bootstrap : $y = r$ (pas de $\gamma Q_{\phi'}$).
- `truncated=True` : limite de temps atteinte. L'etat n'est pas forcement terminal -- le bootstrap doit rester actif.

Dans le training, on stocke `done = float(terminated)` mais on remet l'environnement a zero sur `terminated or truncated`. Pour HalfCheetah-v5, tous les episodes finissent par limite de temps : stocker `done=1` sous-estimerait systematiquement les valeurs de fin d'episode.

## 5. Reseaux cibles et mise a jour douce

### Le probleme des cibles mobiles

Sans reseaux cibles, la cible de Bellman serait :

$$y = r + \gamma Q_\phi(s', \mu_\theta(s'))$$

Mais $Q_\phi$ est le reseau qu'on est en train de mettre a jour ! Chaque gradient modifie $\phi$, ce qui modifie immediatement $y$ -- instabilite garantie.

### La solution : Polyak averaging

DDPG maintient des **copies figees** $\mu_{\theta'}$ et $Q_{\phi'}$, mises a jour lentement apres chaque pas :

$$\theta' \leftarrow \tau \theta + (1-\tau) \theta' \qquad \phi' \leftarrow \tau \phi + (1-\tau) \phi'$$

avec $\tau = 0.005$. Les cibles suivent les reseaux en ligne tres lentement, offrant des targets stables pendant que les reseaux en ligne apprennent rapidement.

> **Note de notation :** SpinningUp utilise parfois $\rho = 1-\tau$ (proche de 1). La relation : $\tau=0.005$ correspond a $\rho=0.995$.

In [ ]:
# === Viz soft update vs hard update ===

fig, ax = plt.subplots(figsize=(9, 5))

n_steps = 100
tau_soft = 0.005
N_hard = 10
steps = np.arange(n_steps)

theta_online = 0.5 * np.sin(0.3 * steps) + steps * 0.005

theta_soft = np.zeros(n_steps)
for t in range(1, n_steps):
    theta_soft[t] = tau_soft * theta_online[t] + (1 - tau_soft) * theta_soft[t - 1]

theta_hard = np.zeros(n_steps)
for t in range(1, n_steps):
    if t % N_hard == 0:
        theta_hard[t] = theta_online[t]
    else:
        theta_hard[t] = theta_hard[t - 1]

ax.plot(steps, theta_online, 'b-', lw=2, label="Reseau en ligne theta", alpha=0.7)
ax.plot(steps, theta_soft, 'r-', lw=2, label=f"Soft update theta' (tau={tau_soft})")
ax.step(steps, theta_hard, 'g--', lw=2,
        label=f"Hard update theta' (tous les {N_hard} pas)", where='post')
ax.set_xlabel('Pas de gradient', fontsize=12)
ax.set_ylabel('Valeur du parametre', fontsize=12)
ax.set_title('Soft update vs Hard update (reseaux cibles)', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

**Interpretation.** La courbe rouge (soft update) suit lentement la bleue (reseau en ligne) : les cibles ne sautent jamais brutalement. La courbe verte (hard update, style DQN) reste constante puis effectue des sauts discontinus.

La mise a jour douce produit des cibles de Bellman qui changent continument mais tres lentement -- le critique apprend donc contre une reference stable.

**Bilan des composants.** Nous avons maintenant :

- **Acteur** $\mu_\theta(s)$ : MLP qui produit une action continue bornee.
- **Critique** $Q_\phi(s,a)$ : MLP qui evalue la qualite d'une action.
- **Buffer** : stocke et reechantillonne les transitions off-policy.
- **Reseaux cibles** $\mu_{\theta'}$, $Q_{\phi'}$ : copies figees pour cibles Bellman stables.

Il manque une piece : comment l'agent explore-t-il, puisque sa politique est deterministe ?

## 6. Le bruit d'exploration

### Pourquoi une politique deterministe explore peu

Une politique deterministe produit toujours la meme action pour le meme etat. La solution : **ajouter du bruit externe** a l'action envoyee a l'environnement.

### Bruit gaussien vs bruit d'Ornstein-Uhlenbeck

Le papier DDPG original propose le **bruit d'Ornstein-Uhlenbeck (OU)**, un processus mean-reverting :

$$x_{t+1} = x_t + \theta_{ou} (\mu - x_t) + \sigma \varepsilon_t, \quad \varepsilon_t \sim \mathcal{N}(0, I)$$

En pratique, le **bruit gaussien i.i.d.** est suffisant et souvent meilleur :

$$\varepsilon_t \sim \mathcal{N}(0, \sigma^2 I)$$

Le papier TD3 (Fujimoto et al., 2018) confirme que le bruit gaussien surpasse OU sur la plupart des benchmarks MuJoCo. Le bruit OU ajoute de la complexite sans gain reel.

In [ ]:
class GaussianNoise:
    """Bruit gaussien i.i.d. -- simple et efficace (recommande par TD3)."""

    def __init__(self, action_dim, sigma=0.1):
        self.action_dim = action_dim
        self.sigma = sigma

    def __call__(self):
        return np.random.normal(0.0, self.sigma, size=self.action_dim).astype(np.float32)

    def reset(self):
        pass  # pas d'etat a reinitialiser


class OUNoise:
    """Processus d'Ornstein-Uhlenbeck -- bruit temporellement correle.

    x_{t+1} = x_t + theta * (mu - x_t) + sigma * N(0, 1)
    Necessite reset() a chaque debut d'episode.
    """

    def __init__(self, action_dim, mu=0.0, theta=0.15, sigma=0.2):
        self.action_dim = action_dim
        self.mu = mu
        self.theta = theta
        self.sigma = sigma
        self._state = np.full(action_dim, mu, dtype=np.float32)

    def __call__(self):
        x = self._state
        dx = self.theta * (self.mu - x) + self.sigma * np.random.randn(self.action_dim)
        self._state = (x + dx).astype(np.float32)
        return self._state.copy()

    def reset(self):
        """Reinitialise l'etat a mu -- appeler au debut de chaque episode."""
        self._state = np.full(self.action_dim, self.mu, dtype=np.float32)

In [ ]:
# === Viz Gaussien vs OU : trajectoires et distributions ===

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

n_steps_noise = 200
sigma = 0.2
np.random.seed(1)

gaussian_noise = np.random.normal(0, sigma, n_steps_noise)

ou_noise = np.zeros(n_steps_noise)
theta_ou = 0.15
for t in range(1, n_steps_noise):
    ou_noise[t] = ou_noise[t-1] + theta_ou * (0 - ou_noise[t-1]) + sigma * np.random.randn()

ax = axes[0]
ax.plot(gaussian_noise, color='royalblue', alpha=0.8, lw=1.2,
        label=f'Bruit gaussien N(0, {sigma}^2)')
ax.plot(ou_noise, color='tomato', alpha=0.8, lw=1.2,
        label=f'Bruit OU (theta={theta_ou}, sigma={sigma})')
ax.axhline(0, color='k', lw=0.5, linestyle='--')
ax.set_xlabel('Pas de temps', fontsize=12)
ax.set_ylabel('Bruit eps_t', fontsize=12)
ax.set_title('Trajectoires : gaussien vs OU', fontsize=13)
ax.legend(fontsize=10)

n_samples = 5000
np.random.seed(2)
g_samples = np.random.normal(0, sigma, n_samples)
ou_samples = np.zeros(n_samples)
for t in range(1, n_samples):
    ou_samples[t] = (ou_samples[t-1] + theta_ou * (0 - ou_samples[t-1])
                     + sigma * np.random.randn())

ax2 = axes[1]
bins = np.linspace(-0.8, 0.8, 50)
ax2.hist(g_samples, bins=bins, density=True, alpha=0.6, color='royalblue', label='Gaussien')
ax2.hist(ou_samples, bins=bins, density=True, alpha=0.6, color='tomato', label='OU')

x_plot = np.linspace(-0.8, 0.8, 300)
sigma_ou_stat = sigma / np.sqrt(2 * theta_ou)
ax2.plot(x_plot, stats.norm.pdf(x_plot, 0, sigma), 'b-', lw=2,
         label=f'N(0, {sigma}^2) theorique')
ax2.plot(x_plot, stats.norm.pdf(x_plot, 0, sigma_ou_stat), 'r--', lw=2,
         label=f'OU stat. N(0, {sigma_ou_stat:.2f}^2)')
ax2.set_xlabel('Valeur du bruit', fontsize=12)
ax2.set_ylabel('Densite', fontsize=12)
ax2.set_title('Distributions marginales', fontsize=13)
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

print(f"Distribution stationnaire OU : sigma_stat = {sigma_ou_stat:.3f}")
print("En pratique, le bruit gaussien simple est suffisant (Fujimoto et al., 2018).")

**Interpretation.** Panneau gauche -- le bruit gaussien saute independamment d'un pas a l'autre (pas de memoire). Le bruit OU forme des trajectoires plus lisses : si le bruit est positif, il a tendance a rester positif avant de revenir vers zero.

Panneau droit -- les deux distributions marginales sont gaussiennes. La distribution stationnaire de OU a un ecart-type $\sigma_{stat} = \sigma / \sqrt{2\theta}$. En pratique, l'avantage de OU disparait sur HalfCheetah : le bruit gaussien simple est suffisant.

### L'astuce du warm-up

Pendant les premiers `start_steps` pas (typiquement 5 000 a 10 000), l'agent echantillonne des actions **uniformement aleatoires** dans l'espace d'action, sans utiliser sa politique.

Cela remplit le buffer avec des transitions diversifiees avant que les mises a jour ne commencent. L'initialisation orthogonale avec gain=0.01 pour l'acteur produit des actions proches de zero -- sans warm-up, le buffer serait biaise vers une region etroite de l'espace d'action.

Nous avons maintenant toutes les briques. Assemblons-les en un agent DDPG complet.

## 7. DDPG : assembler les composants

L'agent DDPG assemble les cinq briques precedentes. Voici les responsabilites de chaque methode :

- **`__init__`** : cree acteur, critique, leurs copies cibles (`copy.deepcopy`), les optimiseurs, le bruit et le buffer.
- **`select_action(obs, deterministic)`** : passe l'observation dans l'acteur, ajoute du bruit si `deterministic=False`, clippe dans les bornes.
- **`store_transition`** : enregistre $(s, a, r, s', \text{done})$ dans le buffer.
- **`learn_step`** : la mise a jour complete en 4 etapes :
  1. Echantillonne un batch
  2. **Critic loss** : $\mathcal{L} = \frac{1}{B}\sum (Q_\phi(s,a) - y)^2$ avec $y = r + \gamma(1-d) Q_{\phi'}(s', \mu_{\theta'}(s'))$
  3. **Actor loss** : $\mathcal{L} = -\frac{1}{B}\sum Q_\phi(s, \mu_\theta(s))$
  4. **Soft update** : $\theta' \leftarrow \tau\theta + (1-\tau)\theta'$ pour les deux cibles

Les methodes `_compute_critic_loss` et `_update_actor_and_targets` sont des **points de fork** : TD3 les surchargera sans toucher au reste de la classe.

In [ ]:
class DDPGAgent:
    """Agent DDPG : acteur deterministe + critique Q + replay buffer + cibles."""

    def __init__(self, obs_dim, action_dim, *, hidden_dim=256,
                 actor_lr=1e-3, critic_lr=1e-3, gamma=0.99, tau=0.005,
                 buffer_capacity=1_000_000, batch_size=256,
                 noise_type="gaussian", noise_std=0.1,
                 action_low=-1.0, action_high=1.0):
        self.gamma = gamma
        self.tau = tau
        self.batch_size = batch_size
        self.action_low = np.asarray(action_low, dtype=np.float32)
        self.action_high = np.asarray(action_high, dtype=np.float32)

        # Reseaux en ligne
        self.actor = DeterministicActor(obs_dim, action_dim, hidden_dim,
                                        action_low, action_high)
        self.critic = ContinuousQNetwork(obs_dim, action_dim, hidden_dim)

        # Reseaux cibles (copies figees, mises a jour en douceur)
        self.actor_target = copy.deepcopy(self.actor)
        self.actor_target.eval()
        self.critic_target = copy.deepcopy(self.critic)
        self.critic_target.eval()

        # Optimiseurs separes
        self.actor_optimizer = optim.Adam(self.actor.parameters(), lr=actor_lr)
        self.critic_optimizer = optim.Adam(self.critic.parameters(), lr=critic_lr)

        # Buffer et bruit
        self.replay_buffer = ContinuousReplayBuffer(buffer_capacity)
        if noise_type == "ou":
            self.noise = OUNoise(action_dim, sigma=noise_std)
        else:
            self.noise = GaussianNoise(action_dim, sigma=noise_std)

    def select_action(self, obs, deterministic=False):
        """mu_theta(s) + bruit si exploration."""
        obs_t = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)
        with torch.no_grad():
            action = self.actor(obs_t).squeeze(0).numpy()
        if not deterministic:
            action = action + self.noise()
            action = np.clip(action, self.action_low, self.action_high)
        return action.astype(np.float32)

    def store_transition(self, state, action, reward, next_state, done):
        """Enregistre (s, a, r, s', done) dans le buffer."""
        self.replay_buffer.push(state, action, reward, next_state, done)
    
    def learn_step(self):
        """Mise a jour DDPG : critic -> actor -> soft update."""
        if len(self.replay_buffer) < self.batch_size:
            return {}

        states, actions, rewards, next_states, dones = \
            self.replay_buffer.sample(self.batch_size)

        # --- Mise a jour du critique ---
        critic_loss = self._compute_critic_loss(
            states, actions, rewards, next_states, dones)
        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        self.critic_optimizer.step()

        # --- Mise a jour de l'acteur + soft updates ---
        actor_metrics = self._update_actor_and_targets(states)

        with torch.no_grad():
            q_mean = self.critic(states, actions).mean().item()

        return {"critic_loss": critic_loss.item(), "q_mean": q_mean, **actor_metrics}

    def _compute_critic_loss(self, states, actions, rewards, next_states, dones):
        """Cible de Bellman DDPG : y = r + gamma*(1-d)*Q'(s', mu'(s'))."""
        with torch.no_grad():
            next_actions = self.actor_target(next_states)
            q_next = self.critic_target(next_states, next_actions)
            q_target = rewards + self.gamma * (1.0 - dones) * q_next
        q_current = self.critic(states, actions)
        return F.mse_loss(q_current, q_target)

    def _update_actor_and_targets(self, states):
        """Actor loss = -E[Q(s, mu(s))] puis soft update."""
        actor_actions = self.actor(states)
        actor_loss = -self.critic(states, actor_actions).mean()

        self.actor_optimizer.zero_grad()
        actor_loss.backward()
        self.actor_optimizer.step()

        # theta' <- tau*theta + (1-tau)*theta'
        self._soft_update(self.actor, self.actor_target, self.tau)
        self._soft_update(self.critic, self.critic_target, self.tau)

        return {"actor_loss": actor_loss.item()}

    @staticmethod
    def _soft_update(online, target, tau):
        """Mise a jour douce des poids cibles."""
        for p_online, p_target in zip(online.parameters(), target.parameters()):
            p_target.data.copy_(tau * p_online.data + (1.0 - tau) * p_target.data)

In [ ]:
# Mini-test agent DDPG : select_action et learn_step

agent_test = DDPGAgent(
    OBS_DIM, ACTION_DIM,
    action_low=ACTION_LOW, action_high=ACTION_HIGH,
    buffer_capacity=1000, batch_size=4,
)

obs_sample = np.zeros(OBS_DIM, dtype=np.float32)
a_det = agent_test.select_action(obs_sample, deterministic=True)
a_exp = agent_test.select_action(obs_sample, deterministic=False)
print(f"select_action (deterministe) : shape={a_det.shape}, val={a_det.round(4)}")
print(f"select_action (exploration)  : shape={a_exp.shape}, val={a_exp.round(4)}")

test_rng = np.random.default_rng(0)
for _ in range(10):
    s = test_rng.standard_normal(OBS_DIM).astype(np.float32)
    a = test_rng.uniform(-1, 1, ACTION_DIM).astype(np.float32)
    r = float(test_rng.standard_normal())
    s2 = test_rng.standard_normal(OBS_DIM).astype(np.float32)
    agent_test.store_transition(s, a, r, s2, False)

metrics = agent_test.learn_step()
print(f"\nlearn_step : {metrics}")
print("Test DDPGAgent : OK")

### Pseudocode DDPG (adapte de SpinningUp)

---

$$\boxed{\begin{aligned}
&\textbf{Algorithme : Deep Deterministic Policy Gradient (DDPG)}\\
&\textbf{Init :} \theta, \phi \text{ aleatoires};\quad \theta' \leftarrow \theta,\; \phi' \leftarrow \phi;\quad \mathcal{D} = \emptyset\\
&\textbf{Repeter :}\\
&\quad 1.\; a = \text{clip}(\mu_\theta(s) + \varepsilon,\; a_{Low},\; a_{High}),\quad \varepsilon \sim \mathcal{N}(0, \sigma)\\
&\quad 2.\; \text{Executer } a,\; \text{observer } s', r, d\\
&\quad 3.\; \text{Stocker } (s, a, r, s', d) \text{ dans } \mathcal{D}\\
&\quad 4.\; \textbf{Si} \text{ c'est le moment de mettre a jour :}\\
&\qquad a.\; \text{Echantillonner batch } B = \{(s, a, r, s', d)\}\\
&\qquad b.\; y = r + \gamma(1-d)\,Q_{\phi'}(s',\, \mu_{\theta'}(s'))\\
&\qquad c.\; \nabla_\phi \frac{1}{|B|}\sum (Q_\phi(s,a) - y)^2 \quad\text{(critic)}\\
&\qquad d.\; \nabla_\theta \frac{1}{|B|}\sum Q_\phi(s,\, \mu_\theta(s)) \quad\text{(actor)}\\
&\qquad e.\; \phi' \leftarrow \tau\phi + (1-\tau)\phi';\quad \theta' \leftarrow \tau\theta + (1-\tau)\theta'
\end{aligned}}$$

---

**Deux points importants.**

1. La cible $y$ utilise les reseaux *cibles* $\phi'$ et $\theta'$, pas les reseaux en ligne.
2. L'acteur suit $\nabla_\theta Q_\phi(s, \mu_\theta(s))$ -- le gradient passe par la composition $Q_\phi \circ \mu_\theta$ (theoreme DPG).

DDPG est maintenant complet. Configurons les hyperparametres et lancons le training.

### Flux DDPG : interaction, critique, acteur et cibles

```mermaid
flowchart LR
    classDef default fill:none,stroke:#64748b,stroke-width:1.5px
    classDef primary fill:none,stroke:#2563eb,stroke-width:2px
    classDef secondary fill:none,stroke:#d97706,stroke-width:2px
    classDef result fill:none,stroke:#16a34a,stroke-width:2px

    env["Environnement"]:::primary
    replay["Replay buffer"]:::primary
    actor["Acteur<br/>$$\mu_\theta(s)$$"]:::primary
    critic["Critique<br/>$$Q_\phi(s,a)$$"]:::secondary
    target["Cibles TD via les réseaux target"]:::secondary
    env -->|"transition réelle"| replay
    actor -->|"action"| env
    replay -->|"batch de transitions"| critic
    replay -->|"même batch"| target
    critic -->|"gradient $$\nabla_\phi$$"| critic_update["Update critique"]:::result
    critic -->|"gradient compose $$\nabla_\theta Q_\phi(s,\mu_\theta(s))$$"| actor_update["Update acteur"]:::result
    actor_update --> polyak["Soft update $$\theta',\phi'$$"]:::result
    critic_update --> polyak
```

In [ ]:
def evaluate(agent, env_id, num_episodes=5, seed=0, max_steps=1000):
    """Evalue la politique deterministe sur plusieurs episodes."""
    eval_env = gym.make(env_id)
    rewards = []
    try:
        for episode_idx in range(num_episodes):
            obs, _ = eval_env.reset(seed=seed + episode_idx)
            total_reward = 0.0
            for _ in range(max_steps):
                action = agent.select_action(obs, deterministic=True)
                obs, reward, terminated, truncated, _ = eval_env.step(action)
                total_reward += reward
                if terminated or truncated:
                    break
            rewards.append(total_reward)
    finally:
        eval_env.close()
    return float(np.mean(rewards)), float(np.std(rewards))


def _empty_history(include_twin_metrics=False):
    history = {
        "episode_rewards": [],
        "episode_steps": [],
        "eval_steps": [],
        "eval_mean_rewards": [],
        "eval_std_rewards": [],
        "step_update_steps": [],
        "step_critic_losses": [],
        "step_actor_update_steps": [],
        "step_actor_losses": [],
        "step_q_means": [],
    }
    if include_twin_metrics:
        history.update(
            {
                "step_q1_means": [],
                "step_q2_means": [],
                "step_q_gaps": [],
                "step_actor_update_flags": [],
            }
        )
    return history

In [ ]:
def record_offpolicy_update(history, metrics, total_steps, include_twin_metrics):
    """Enregistre les diagnostics sans cacher le learn_step algorithmique."""
    critic_loss = metrics["critic_loss"]
    q_mean = metrics["q_mean"]
    actor_loss = metrics.get("actor_loss", float("nan"))
    actor_updated = bool(metrics.get("actor_updated", np.isfinite(actor_loss)))

    history["step_update_steps"].append(total_steps)
    history["step_critic_losses"].append(critic_loss)
    history["step_q_means"].append(q_mean)
    if actor_updated and np.isfinite(actor_loss):
        history["step_actor_update_steps"].append(total_steps)
        history["step_actor_losses"].append(actor_loss)

    q1_mean = q2_mean = q_gap = float("nan")
    if include_twin_metrics:
        q1_mean = metrics.get("q1_mean", float("nan"))
        q2_mean = metrics.get("q2_mean", float("nan"))
        q_gap = metrics.get("q_gap", float("nan"))
        history["step_q1_means"].append(q1_mean)
        history["step_q2_means"].append(q2_mean)
        history["step_q_gaps"].append(q_gap)
        history["step_actor_update_flags"].append(float(actor_updated))

    return {
        "critic_loss": critic_loss,
        "actor_loss": actor_loss if actor_updated else float("nan"),
        "actor_updated": actor_updated and np.isfinite(actor_loss),
        "q_mean": q_mean,
        "q1_mean": q1_mean,
        "q2_mean": q2_mean,
        "q_gap": q_gap,
    }


def update_offpolicy_progress(progress, **view):
    """Met uniquement en forme la barre de progression et ses diagnostics."""
    rewards = view["history"]["episode_rewards"]
    avg20 = float(np.mean(rewards[-20:])) if rewards else float("nan")
    elapsed = time.perf_counter() - view["start_time"]
    postfix = {
        "phase": view["phase"],
        "ep": view["episode"],
        "rew": f"{view['episode_reward']:.1f}",
        "avg20": f"{avg20:.1f}",
        "crit": f"{view['critic_loss']:.3f}",
        "act": f"{view['actor_loss']:.3f}",
        "eval": f"{view['eval_mean']:.1f}",
        "spd": f"{view['total_steps'] / max(elapsed, 1e-8):.0f}/s",
    }
    if view["include_twin_metrics"]:
        postfix.update(
            q1=f"{view['q1_mean']:.2f}",
            q2=f"{view['q2_mean']:.2f}",
            gap=f"{view['q_gap']:.3f}",
        )
    else:
        postfix["q"] = f"{view['q_mean']:.2f}"
    progress.set_postfix(refresh=False, **postfix)



def run_offpolicy_evaluation(agent, history, *, label, total_steps, q_gap, include_twin_metrics):
    """Évalue, enregistre et affiche un checkpoint sous un protocole fixe."""
    eval_mean, eval_std = evaluate(
        agent, ENV_ID, EVAL_EPISODES, seed=EVAL_SEED, max_steps=MAX_STEPS_PER_EPISODE
    )
    history["eval_steps"].append(total_steps)
    history["eval_mean_rewards"].append(eval_mean)
    history["eval_std_rewards"].append(eval_std)
    q_gap_msg = f" | Q gap={q_gap:.3f}" if include_twin_metrics else ""
    tqdm.write(
        f"[{label}] step {total_steps:>7,d} | "
        f"eval={eval_mean:>8.1f} +/- {eval_std:.1f}{q_gap_msg}"
    )
    return eval_mean


def print_offpolicy_summary(history, *, label, episode, total_steps, actor_updates, elapsed, twin):
    """Résume le run sans alourdir le flot env -> replay -> learn_step."""
    best_eval = max(history["eval_mean_rewards"]) if history["eval_mean_rewards"] else float("nan")
    detail = f"actor updates={actor_updates:,}" if twin else f"steps={total_steps:,}"
    print(
        f"\n=== {label} termine === episodes={episode} | {detail} | "
        f"duree={elapsed/60:.1f}min | meilleure eval={best_eval:.1f}"
    )


In [ ]:
def train_offpolicy_agent(agent, *, label, include_twin_metrics=False):
    """Boucle commune DDPG/TD3 : l'agent garde les maths, la boucle orchestre."""
    if tqdm is None:
        raise RuntimeError("tqdm est requis pour cette cellule de training.")

    np.random.seed(SEED)
    torch.manual_seed(SEED)

    env = gym.make(ENV_ID)
    env.action_space.seed(SEED)
    history = _empty_history(include_twin_metrics=include_twin_metrics)

    total_steps = 0
    episode = 0
    actor_update_count = 0
    warmup_announced = False
    last_episode_reward = float("nan")
    last_critic_loss = float("nan")
    last_actor_loss = float("nan")
    last_q_mean = float("nan")
    last_q1_mean = float("nan")
    last_q2_mean = float("nan")
    last_q_gap = float("nan")
    last_eval_mean = float("nan")
    start_time = time.perf_counter()

    print(f"=== {label} training ===")
    print(f"Total steps : {TOTAL_STEPS:,} | Warm-up : {START_STEPS:,} | Seed : {SEED}")
    if include_twin_metrics:
        print(
            f"policy_delay={agent.policy_delay} | "
            f"target_noise={agent.target_noise}"
        )

    progress = tqdm(total=TOTAL_STEPS, desc=label, unit="step", mininterval=0.5)
    try:
        while total_steps < TOTAL_STEPS:
            obs, _ = env.reset(seed=SEED + episode)
            episode_reward = 0.0
            agent.noise.reset()

            for _ in range(MAX_STEPS_PER_EPISODE):
                if total_steps < START_STEPS:
                    action = env.action_space.sample()
                    phase = "warmup"
                else:
                    action = agent.select_action(obs, deterministic=False)
                    phase = "learning"

                next_obs, reward, terminated, truncated, _ = env.step(action)
                episode_done = terminated or truncated

                agent.store_transition(obs, action, reward, next_obs, float(terminated))
                episode_reward += reward
                obs = next_obs
                total_steps += 1
                progress.update(1)

                if total_steps >= UPDATE_AFTER:
                    metrics = agent.learn_step()
                    if metrics:
                        update_view = record_offpolicy_update(
                            history, metrics, total_steps, include_twin_metrics
                        )
                        last_critic_loss = update_view["critic_loss"]
                        last_actor_loss = update_view["actor_loss"]
                        last_q_mean = update_view["q_mean"]
                        last_q1_mean = update_view["q1_mean"]
                        last_q2_mean = update_view["q2_mean"]
                        last_q_gap = update_view["q_gap"]
                        actor_update_count += int(update_view["actor_updated"])

                if total_steps >= START_STEPS and not warmup_announced:
                    tqdm.write(f"[{label}] Warm-up termine a {total_steps:,} steps.")
                    warmup_announced = True

                if total_steps % EVAL_EVERY == 0:
                    last_eval_mean = run_offpolicy_evaluation(
                        agent,
                        history,
                        label=label,
                        total_steps=total_steps,
                        q_gap=last_q_gap,
                        include_twin_metrics=include_twin_metrics,
                    )

                if total_steps % POSTFIX_EVERY_STEPS == 0:
                    update_offpolicy_progress(
                        progress,
                        history=history,
                        phase=phase,
                        episode=episode,
                        total_steps=total_steps,
                        start_time=start_time,
                        episode_reward=last_episode_reward,
                        critic_loss=last_critic_loss,
                        actor_loss=last_actor_loss,
                        eval_mean=last_eval_mean,
                        q_mean=last_q_mean,
                        q1_mean=last_q1_mean,
                        q2_mean=last_q2_mean,
                        q_gap=last_q_gap,
                        include_twin_metrics=include_twin_metrics,
                    )

                if episode_done or total_steps >= TOTAL_STEPS:
                    break

            last_episode_reward = float(episode_reward)
            history["episode_rewards"].append(last_episode_reward)
            history["episode_steps"].append(total_steps)
            episode += 1

            if episode % LOG_EVERY_EPISODES == 0:
                avg20 = float(np.mean(history["episode_rewards"][-20:]))
                if include_twin_metrics:
                    tqdm.write(
                        f"[{label}] ep {episode:>4d} | steps {total_steps:>7,d} | "
                        f"rew {episode_reward:>8.1f} | avg20 {avg20:>8.1f} | "
                        f"actor updates {actor_update_count:>6,d}"
                    )
                else:
                    tqdm.write(
                        f"[{label}] ep {episode:>4d} | steps {total_steps:>7,d} | "
                        f"rew {episode_reward:>8.1f} | avg20 {avg20:>8.1f}"
                    )
    finally:
        progress.close()
        env.close()

    elapsed = time.perf_counter() - start_time
    print_offpolicy_summary(
        history,
        label=label,
        episode=episode,
        total_steps=total_steps,
        actor_updates=actor_update_count,
        elapsed=elapsed,
        twin=include_twin_metrics,
    )
    return history

In [ ]:
def evaluate_and_display_agent(
    agent,
    label,
    n_episodes=3,
    seed=DEMO_SEED,
    max_steps=MAX_STEPS_PER_EPISODE,
    video_root="videos/07_ddpg_td3_halfcheetah",
):
    """Évalue une politique déterministe et affiche le dernier replay vidéo enregistré."""
    safe_label = label.lower().replace(" ", "_").replace("-", "_")
    video_dir = Path(video_root) / safe_label
    video_dir.mkdir(parents=True, exist_ok=True)

    video_env = gym.make(ENV_ID, render_mode="rgb_array")
    video_env = RecordVideo(
        video_env,
        video_folder=str(video_dir),
        episode_trigger=lambda episode_id: episode_id == n_episodes - 1,
        name_prefix=f"{safe_label}_greedy",
    )

    rewards = []
    try:
        for episode_idx in range(n_episodes):
            obs, _ = video_env.reset(seed=seed + episode_idx)
            total_reward = 0.0
            steps = 0

            for _ in range(max_steps):
                action = agent.select_action(obs, deterministic=True)
                obs, reward, terminated, truncated, _ = video_env.step(action)
                total_reward += float(reward)
                steps += 1
                if terminated or truncated:
                    break

            rewards.append(total_reward)
            print(f"[{label}] Épisode {episode_idx + 1} : retour={total_reward:+.1f} ({steps} pas)")

    finally:
        video_env.close()

    print(f"[{label}] Moyenne : {np.mean(rewards):+.1f} +/- {np.std(rewards):.1f}")

    videos = sorted(video_dir.glob("*.mp4"), key=lambda path: path.stat().st_mtime)
    if videos and Video is not None and display is not None:
        print(f"Replay vidéo {label} :", videos[-1])
        display(Video(str(videos[-1]), embed=True, width=640))
    elif not videos:
        print(f"Aucune vidéo générée pour {label}.")
    else:
        print(f"Vidéo générée mais affichage IPython indisponible : {videos[-1]}")

    return rewards


In [ ]:
# Configuration commune DDPG / TD3 : seules les équations internes des agents diffèrent.
SEED = 42
ENV_ID = "HalfCheetah-v5"
EVAL_SEED = SEED
DEMO_SEED = SEED

TOTAL_STEPS = 100_000
START_STEPS = 5_000
UPDATE_AFTER = 1_000
EVAL_EVERY = 10_000
EVAL_EPISODES = 5
LOG_EVERY_EPISODES = 20
POSTFIX_EVERY_STEPS = 250
MAX_STEPS_PER_EPISODE = 1000

torch.manual_seed(SEED)
np.random.seed(SEED)


In [ ]:
# --- Boucle d'entrainement DDPG ---

agent = DDPGAgent(
    OBS_DIM, ACTION_DIM,
    buffer_capacity=100_000, batch_size=256,
    actor_lr=1e-3, critic_lr=1e-3,
    gamma=0.99, tau=0.005,
    noise_type="gaussian", noise_std=0.1,
    action_low=ACTION_LOW, action_high=ACTION_HIGH,
)
print(f"Agent DDPG cree. Buffer capacity={100_000:,} | Seed={SEED}")
ddpg_history = train_offpolicy_agent(agent, label="DDPG", include_twin_metrics=False)


DDPG est entraine. Avant de passer a TD3, prenons le temps d'identifier pourquoi DDPG peut echouer -- ce sont exactement les problemes que TD3 va resoudre.

**DDPG fonctionne -- mais il souffre de trois problemes structurels qui le rendent instable en pratique.**

## 9. Probleme 1 : le critique surestime les valeurs Q

Le critique $Q_\phi(s, a)$ a tendance a **surestimer** les vraies valeurs. Ce phenomene est connu depuis DQN, mais il est encore plus destructeur dans le cas deterministe.

### Pourquoi ca arrive

L'erreur d'approximation du reseau produit des valeurs Q bruitees. Quand on optimise par rapport a l'action, on selectionne systematiquement les erreurs positives :

$$\mathbb{E}\left[\max_a Q(s,a)\right] \geq \max_a \mathbb{E}\left[Q(s,a)\right] \quad \text{(inegalite de Jensen)}$$

### Pourquoi c'est pire dans DDPG que dans DQN

Dans DDPG, **l'acteur apprend explicitement a exploiter les erreurs du critique** : il ajuste $\theta$ pour maximiser $Q_\phi(s, \mu_\theta(s))$. Si $Q_\phi$ surevalue une action, l'acteur converge vers cette action -- pas parce qu'elle est bonne, mais parce que le critique la juge bonne par erreur.

Le cercle vicieux : acteur vers actions surevaluees -> buffer biaise -> critique continue de surevaluer -> acteur diverge davantage.

In [ ]:
# === Viz overestimation du critique et accumulation de biais ===

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

np.random.seed(7)
a_range = np.linspace(-1.0, 1.0, 300)

Q_true = -(a_range - 0.3)**2 + 1.0
noise_level = 0.25
Q_hat = Q_true + noise_level * np.random.randn(len(a_range))
Q_hat = uniform_filter1d(Q_hat, size=10) + 0.15

a_true_opt = a_range[np.argmax(Q_true)]
a_estimated_opt = a_range[np.argmax(Q_hat)]

ax = axes[0]
ax.plot(a_range, Q_true, 'b-', lw=2, label='Q vrai(s,a)')
ax.plot(a_range, Q_hat, 'r-', lw=2, alpha=0.9, label='Q phi(s,a) estime')
ax.axvline(a_true_opt, color='b', linestyle='--', lw=1.5,
           label=f'a* optimal = {a_true_opt:.2f}')
ax.axvline(a_estimated_opt, color='r', linestyle='--', lw=1.5,
           label=f'Acteur choisi = {a_estimated_opt:.2f}')
ax.annotate('', xy=(a_estimated_opt, Q_true[np.argmin(np.abs(a_range - a_estimated_opt))]),
    xytext=(a_true_opt, Q_true[np.argmax(Q_true)]),
    arrowprops=dict(arrowstyle='->', color='purple', lw=2.0))
ax.text((a_true_opt + a_estimated_opt)/2, 0.6, 'Exploitation\ndes erreurs',
        ha='center', color='purple', fontsize=10)
ax.set_xlabel('Action a')
ax.set_ylabel('Q(s, a)')
ax.set_title("Overestimation : l'acteur exploite les erreurs du critique")
ax.legend(fontsize=9)

n_updates = 200
np.random.seed(3)
updates = np.arange(n_updates)
overest_ddpg = 0.5 + 0.015 * updates + 0.3 * np.cumsum(np.random.randn(n_updates) * 0.1)
overest_ddpg = np.clip(overest_ddpg, 0, None)
overest_td3 = 0.3 + 0.002 * updates + 0.15 * np.cumsum(np.random.randn(n_updates) * 0.05)
overest_td3 = np.clip(overest_td3, 0, None)

ax2 = axes[1]
ax2.plot(updates, overest_ddpg, 'b-', alpha=0.8, lw=1.5, label='DDPG : biais croissant')
ax2.plot(updates, overest_td3, 'r-', alpha=0.8, lw=1.5, label='TD3 : biais attenue par min(Q1, Q2)')
ax2.set_xlabel('Nombre de mises a jour')
ax2.set_ylabel("Overestimation Q_hat - Q*")
ax2.set_title("Evolution du biais d'overestimation (simulation)")
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

**Interpretation.** Panneau gauche : le critique estime a un biais positif global et un pic artificiel. L'acteur converge vers $a \approx 0.56$ (ligne rouge) plutot que vers l'optimum reel $a^* = 0.30$ (bleu) -- il exploite une erreur du critique.

Panneau droit : simulation de l'accumulation du biais. DDPG accumule l'overestimation progressivement (cercle vicieux acteur-critique). TD3 le plafonne grace au minimum de deux critiques independants -- anticipons la correction.

## 9b. Problemes 2 et 3 de DDPG (brefs)

**Probleme 2 : fragilite de l'acteur face a un critique bruite.**

Le gradient acteur traverse directement $Q_\phi$ : $\nabla_\theta \mathcal{L}_{actor} = -\nabla_a Q_\phi \cdot \nabla_\theta \mu_\theta$. Si $Q_\phi$ a des pics ou irregularites locales, l'acteur suit ces gradients aveuglement. Un seul gradient errone peut produire une mise a jour catastrophique.

**Probleme 3 : couplage acteur-critique.**

L'acteur et le critique sont mis a jour a chaque pas. Ce cycle de retroaction rapide :

$$\theta \to \mu_\theta \to Q_{\phi'} \to y_i \to Q_\phi \to \mathcal{L}_{actor} \to \theta$$

genere des oscillations qui s'accumulent sur le long terme. TD3 adresse les trois problemes avec trois corrections chirurgicales.

## 10. TD3 -- Correction 1 : Twin critics

### L'idee : la prudence pessimiste

TD3 entraine **deux critiques independants** $Q_{\phi_1}$ et $Q_{\phi_2}$ et utilise leur **minimum** pour calculer la cible de Bellman.

La cible DDPG etait :

$$y = r + \gamma (1 - d) \cdot Q_{\phi'}(s', \mu_{\theta'}(s'))$$

La cible TD3 devient :

$$\boxed{y = r + \gamma (1 - d) \cdot \min\left(Q_{\phi_1'}(s', \tilde{a}'),\, Q_{\phi_2'}(s', \tilde{a}')\right)}$$

### Pourquoi le minimum fonctionne

Les deux critiques ont des initialisations et trajectoires d'entrainement differentes. Leur biais commun est l'overestimation (biais positif). En prenant le minimum, on favorise la valeur **la plus pessimiste** -- ce pessimisme compense le biais positif naturel.

Les deux critiques sont entraines simultanement avec la meme cible $y$ :

$$\mathcal{L}_{critic} = \frac{1}{B}\sum_i \left[(Q_{\phi_1}(s_i, a_i) - y_i)^2 + (Q_{\phi_2}(s_i, a_i) - y_i)^2\right]$$

Pour la perte de l'acteur, TD3 n'utilise que $Q_{\phi_1}$ (par convention).

## 11. TD3 -- Correction 2 : Delayed policy updates

### Le probleme : couplage acteur-critique

Dans DDPG, l'acteur et le critique sont mis a jour a chaque pas. L'acteur est entraine a maximiser $Q_\phi$ alors que $Q_\phi$ est encore bruite. Resultat : l'acteur fait des pas bases sur un gradient bruite, degrade le critique...

### La solution : mettre a jour l'acteur moins souvent

TD3 introduit un **delai de politique** : l'acteur (et les reseaux cibles) n'est mis a jour que tous les $d$ pas du critique ($d = 2$ par defaut).

```
Pour chaque pas t :
    Mise a jour du critique  (toujours)
    Si t % d == 0 :
        Mise a jour de l'acteur
        Soft update des reseaux cibles
```

**Intuition.** Laisser le critique converger un peu avant de bouger l'acteur reduit la variance du gradient acteur. C'est analogue a l'idee de PPO : ne pas changer la politique trop vite par rapport a l'evaluation de la politique.

En pratique, $d = 2$ suffit pour une amelioration notable. Des valeurs plus grandes ralentissent l'apprentissage sans benefice supplementaire.

## 12. TD3 -- Correction 3 : Target policy smoothing

### Le probleme des pics artificiels

Le critique $Q_\phi$ peut avoir des **pics artificiels** -- des actions ou $Q_\phi$ est tres eleve sans justification reelle. Si l'acteur est assez flexible, il converge vers ces pics.

### La solution : bruiter l'action cible

TD3 ajoute du bruit **clippe** a l'action cible utilisee dans la cible de Bellman :

$$\tilde{a}' = \text{clip}\!\left(\mu_{\theta'}(s') + \text{clip}(\varepsilon, -c, c),\; a_{\min},\; a_{\max}\right), \quad \varepsilon \sim \mathcal{N}(0, \sigma_{target}^2)$$

La cible complete TD3 devient :

$$\boxed{y = r + \gamma (1 - d) \cdot \min\left(Q_{\phi_1'}(s', \tilde{a}'),\, Q_{\phi_2'}(s', \tilde{a}')\right)}$$

En ajoutant du bruit a l'action cible, on force le critique a apprendre des valeurs Q **localement consistantes**. Le clipping $[-c, c]$ empeche que le bruit ne deplace l'action vers une region hors distribution du buffer.

## TD3 = DDPG + 3 corrections chirurgicales

TD3 herite de `DDPGAgent` et ne change que les points ou DDPG etait le plus fragile.
On peut le lire comme trois fourches locales dans le meme pipeline.

| Correction | Methode modifiee | Effet |
|------------|------------------|-------|
| Twin critics | `_compute_critic_loss` | remplace une unique cible par `min(Q1', Q2')` |
| Target smoothing | `_compute_critic_loss` | perturbe l'action cible avant Bellman |
| Delayed updates | `_update_actor_and_targets` | ralentit l'acteur et les cibles |

```mermaid
flowchart TB
    classDef default fill:none,stroke:#64748b,stroke-width:1.5px
    classDef primary fill:none,stroke:#2563eb,stroke-width:2px
    classDef secondary fill:none,stroke:#d97706,stroke-width:2px
    classDef result fill:none,stroke:#16a34a,stroke-width:2px

    batch["Replay batch"]:::primary
    twin["Twin critics<br/>$$Q_{\phi_1}, Q_{\phi_2}$$"]:::secondary
    target_actor["Target actor<br/>$$\mu_{\theta'}(s')$$"]:::primary
    smooth["Action cible lissée"]:::secondary
    batch --> twin
    target_actor --> smooth
    noise["Bruit clippe $$\epsilon \sim \mathrm{clip}(\mathcal{N}(0,\sigma),-c,c)$$"]:::secondary --> smooth
    smooth --> target_min["Cible $$\min(Q_{\phi_1'},Q_{\phi_2'})(s',\tilde{a}')$$"]:::secondary
    twin --> critic_step["Update critiques a chaque pas"]:::result
    target_min --> critic_step
    critic_step --> delay["Attendre $$policy\_delay$$ pas"]:::result
    delay --> actor_step["Update acteur avec $$Q_{\phi_1}$$"]:::result
    actor_step --> polyak["Soft update des cibles"]:::result
```

In [ ]:
class TwinQNetwork(nn.Module):
    """Paire de Q-networks independants pour TD3.

    Deux ContinuousQNetwork avec poids independants.
    La cible Bellman utilise min(Q1, Q2) pour reduire le biais positif.
    """

    def __init__(self, obs_dim, action_dim, hidden_dim=256):
        super().__init__()
        self.q1 = ContinuousQNetwork(obs_dim, action_dim, hidden_dim)
        self.q2 = ContinuousQNetwork(obs_dim, action_dim, hidden_dim)

    def forward(self, state, action):
        """Retourne (Q1(s,a), Q2(s,a))."""
        return self.q1(state, action), self.q2(state, action)

    def q1_forward(self, state, action):
        """Q1 seul -- utilise pour la loss acteur (evite de calculer Q2)."""
        return self.q1(state, action)

In [ ]:
class TD3Agent(DDPGAgent):
    """Agent TD3 -- DDPG + twin critics + delayed updates + target smoothing."""

    def __init__(self, obs_dim, action_dim, *, policy_delay=2,
                 target_noise=0.2, target_noise_clip=0.5, **kwargs):
        super().__init__(obs_dim, action_dim, **kwargs)

        self.critic = TwinQNetwork(obs_dim, action_dim, kwargs.get("hidden_dim", 256))
        self.critic_target = copy.deepcopy(self.critic)
        self.critic_target.eval()
        self.critic_optimizer = optim.Adam(
            self.critic.parameters(), lr=kwargs.get("critic_lr", 1e-3)
        )

        self.policy_delay = policy_delay
        self.target_noise = target_noise
        self.target_noise_clip = target_noise_clip
        self._update_step = 0

        self._action_low_t = torch.tensor(self.action_low)
        self._action_high_t = torch.tensor(self.action_high)

    def _compute_critic_loss(self, states, actions, rewards, next_states, dones):
        """Twin critics + target policy smoothing."""
        with torch.no_grad():
            next_actions = self.actor_target(next_states)
            noise = (torch.randn_like(next_actions) * self.target_noise).clamp(
                -self.target_noise_clip, self.target_noise_clip
            )
            next_actions = (next_actions + noise).clamp(self._action_low_t, self._action_high_t)
            q1_next, q2_next = self.critic_target(next_states, next_actions)
            q_next = torch.min(q1_next, q2_next)
            q_target = rewards + self.gamma * (1.0 - dones) * q_next

        q1, q2 = self.critic(states, actions)
        return F.mse_loss(q1, q_target) + F.mse_loss(q2, q_target)

    def _update_actor_and_targets(self, states):
        """Delayed policy update : acteur mis a jour tous les policy_delay pas."""
        self._update_step += 1
        if self._update_step % self.policy_delay != 0:
            return {"actor_loss": float("nan"), "actor_updated": False}

        actor_actions = self.actor(states)
        actor_loss = -self.critic.q1_forward(states, actor_actions).mean()

        self.actor_optimizer.zero_grad()
        actor_loss.backward()
        self.actor_optimizer.step()

        self._soft_update(self.actor, self.actor_target, self.tau)
        self._soft_update(self.critic, self.critic_target, self.tau)
        return {"actor_loss": actor_loss.item(), "actor_updated": True}

    def learn_step(self):
        """Mise a jour TD3 avec metriques enrichies (q1, q2, q_gap)."""
        if len(self.replay_buffer) < self.batch_size:
            return {}

        states, actions, rewards, next_states, dones = self.replay_buffer.sample(self.batch_size)
        critic_loss = self._compute_critic_loss(states, actions, rewards, next_states, dones)
        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        self.critic_optimizer.step()

        actor_metrics = self._update_actor_and_targets(states)

        with torch.no_grad():
            q1, q2 = self.critic(states, actions)
            q1_mean, q2_mean = q1.mean().item(), q2.mean().item()

        return {
            "critic_loss": critic_loss.item(),
            "q_mean": (q1_mean + q2_mean) / 2,
            "q1_mean": q1_mean,
            "q2_mean": q2_mean,
            "q_gap": (q1 - q2).abs().mean().item(),
            **actor_metrics,
        }

In [ ]:
# Mini-test TD3Agent : select_action, learn_step, policy_delay

agent_td3_test = TD3Agent(
    OBS_DIM, ACTION_DIM,
    action_low=ACTION_LOW, action_high=ACTION_HIGH,
    buffer_capacity=1000, batch_size=4,
    policy_delay=2,
)

test_rng = np.random.default_rng(1)
for _ in range(10):
    s = test_rng.standard_normal(OBS_DIM).astype(np.float32)
    a = test_rng.uniform(-1, 1, ACTION_DIM).astype(np.float32)
    r = float(test_rng.standard_normal())
    s2 = test_rng.standard_normal(OBS_DIM).astype(np.float32)
    agent_td3_test.store_transition(s, a, r, s2, False)

print("Updates avec policy_delay=2 :")
for step in range(1, 7):
    m = agent_td3_test.learn_step()
    actor_updated = m.get("actor_updated", False)
    print(f"  Step {step} | critic_loss={m['critic_loss']:.4f} | actor_updated={actor_updated}")

print("\nTest TD3Agent : OK -- acteur mis a jour tous les 2 pas")

In [ ]:
# --- Boucle d'entrainement TD3 ---
agent_td3 = TD3Agent(
    OBS_DIM, ACTION_DIM,
    buffer_capacity=100_000, batch_size=256,
    actor_lr=1e-3, critic_lr=1e-3,
    gamma=0.99, tau=0.005,
    noise_type="gaussian", noise_std=0.1,
    action_low=ACTION_LOW, action_high=ACTION_HIGH,
    policy_delay=2, target_noise=0.2, target_noise_clip=0.5,
)
print(f"Agent TD3 cree. policy_delay={agent_td3.policy_delay} | Seed={SEED}")
td3_history = train_offpolicy_agent(agent_td3, label="TD3", include_twin_metrics=True)


# Les diagnostics TD3 ci-dessous lisent `td3_history` produit par la boucle commune.

In [ ]:
# === Helpers pour la comparaison ===

def paired_series(history, x_key, y_key):
    """Retourne deux tableaux alignes, finis."""
    y = np.asarray(history.get(y_key, []), dtype=float)
    x_values = history.get(x_key, [])
    x = (np.asarray(x_values, dtype=float) if len(x_values)
         else np.arange(len(y), dtype=float))
    n = min(len(x), len(y))
    x, y = x[:n], y[:n]
    finite = np.isfinite(x) & np.isfinite(y)
    return x[finite], y[finite]


def moving_average_xy(x, y, window=10):
    """Lisse une serie sans decaler son axe x."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(y) < window:
        return x, y
    kernel = np.ones(window, dtype=float) / window
    return x[window - 1:], np.convolve(y, kernel, mode="valid")


def plot_smoothed(ax, x, y, *, color, label, window, **kwargs):
    if len(y) == 0:
        return
    x_smooth, y_smooth = moving_average_xy(x, y, window)
    ax.plot(x_smooth, y_smooth, color=color, label=label, **kwargs)


def legend_if_present(ax):
    handles, _ = ax.get_legend_handles_labels()
    if handles:
        ax.legend()


def validate_history_alignment(name, history):
    pairs = [
        ("episode_steps", "episode_rewards"),
        ("step_update_steps", "step_critic_losses"),
        ("step_update_steps", "step_q_means"),
        ("step_actor_update_steps", "step_actor_losses"),
    ]
    for x_key, y_key in pairs:
        x_len = len(history.get(x_key, []))
        y_len = len(history.get(y_key, []))
        if x_len and x_len != y_len:
            print(f"Attention {name}: {x_key}={x_len}, {y_key}={y_len}")

In [ ]:
# === Comparaison DDPG vs TD3 -- 6 panneaux ===

if ddpg_history is None or td3_history is None:
    print("Executer les trainings DDPG et TD3 avant cette cellule.")
else:
    ddpg_h = ddpg_history
    td3_h = td3_history
    validate_history_alignment("DDPG", ddpg_h)
    validate_history_alignment("TD3", td3_h)

    color_ddpg = "#1f77b4"
    color_td3 = "#ff7f0e"
    reward_window = 15
    metric_window = 200

    fig, axes = plt.subplots(2, 3, figsize=(19, 11))

    # 1. Recompenses d'entrainement
    ax = axes[0, 0]
    for history, color, name in [(ddpg_h, color_ddpg, "DDPG"), (td3_h, color_td3, "TD3")]:
        x, rewards = paired_series(history, "episode_steps", "episode_rewards")
        if len(rewards):
            ax.plot(x, rewards, color=color, alpha=0.12)
            plot_smoothed(ax, x, rewards, color=color,
                          label=f"{name} moy. {reward_window}", window=reward_window, lw=2)
    ax.set_title("Recompenses d'entrainement (politique exploratoire)")
    ax.set_xlabel("Environment steps")
    ax.set_ylabel("Reward par episode")
    legend_if_present(ax)

    # 2. Evaluation deterministe (metrique principale)
    ax = axes[0, 1]
    for history, color, name in [(ddpg_h, color_ddpg, "DDPG"), (td3_h, color_td3, "TD3")]:
        steps, means = paired_series(history, "eval_steps", "eval_mean_rewards")
        stds = np.asarray(history.get("eval_std_rewards", []), dtype=float)[:len(means)]
        if len(means):
            ax.plot(steps, means, marker="o", color=color, label=name)
            if len(stds) == len(means):
                ax.fill_between(steps, means - stds, means + stds, color=color, alpha=0.15)
    ax.set_title("Evaluation deterministe -- metrique principale")
    ax.set_xlabel("Environment steps")
    ax.set_ylabel("Reward moyen +/- std")
    legend_if_present(ax)

    # 3. Valeurs Q
    ax = axes[0, 2]
    for history, color, name in [(ddpg_h, color_ddpg, "DDPG Q"),
                                   (td3_h, color_td3, "TD3 mean(Q1,Q2)")]:
        x, q_values = paired_series(history, "step_update_steps", "step_q_means")
        plot_smoothed(ax, x, q_values, color=color, label=name, window=metric_window, lw=1.8)
    for key, ls, alpha, label in [("step_q1_means", "--", 0.45, "TD3 Q1"),
                                    ("step_q2_means", ":", 0.45, "TD3 Q2")]:
        x, values = paired_series(td3_h, "step_update_steps", key)
        plot_smoothed(ax, x, values, color=color_td3, label=label,
                      window=metric_window, lw=1.2, linestyle=ls, alpha=alpha)
    ax.set_title("Valeurs Q apprises")
    ax.set_xlabel("Environment steps")
    ax.set_ylabel("Q moyen")
    legend_if_present(ax)

    # 4. Critic loss
    ax = axes[1, 0]
    ddpg_steps, ddpg_loss = paired_series(ddpg_h, "step_update_steps", "step_critic_losses")
    td3_steps, td3_loss_sum = paired_series(td3_h, "step_update_steps", "step_critic_losses")
    plot_smoothed(ax, ddpg_steps, ddpg_loss, color=color_ddpg,
                  label="DDPG critic MSE", window=metric_window, lw=1.8)
    plot_smoothed(ax, td3_steps, td3_loss_sum / 2.0, color=color_td3,
                  label="TD3 MSE moy. par critic", window=metric_window, lw=1.8)
    ax.set_title("Erreur de Bellman (comparable)")
    ax.set_xlabel("Environment steps")
    ax.set_ylabel("MSE moy. par critic")
    ax.set_yscale("symlog", linthresh=1e-4)
    legend_if_present(ax)

    # 5. Actor loss
    ax = axes[1, 1]
    for history, color, name in [(ddpg_h, color_ddpg, "DDPG actor"),
                                   (td3_h, color_td3, "TD3 actor (delayed)")]:
        x, losses = paired_series(history, "step_actor_update_steps", "step_actor_losses")
        plot_smoothed(ax, x, losses, color=color, label=name, window=metric_window, lw=1.8)
    ax.set_title("Actor loss (aux vraies updates acteur)")
    ax.set_xlabel("Environment steps")
    ax.set_ylabel("-E[Q(s, mu(s))]")
    legend_if_present(ax)

    # 6. Q gap TD3
    ax = axes[1, 2]
    gap_steps, q_gap = paired_series(td3_h, "step_update_steps", "step_q_gaps")
    plot_smoothed(ax, gap_steps, q_gap, color=color_td3,
                  label="TD3 mean |Q1-Q2|", window=metric_window, lw=1.8)
    ax.set_title("Desaccord entre les critics TD3")
    ax.set_xlabel("Environment steps")
    ax.set_ylabel("Mean |Q1-Q2|")
    legend_if_present(ax)

    for ax in axes.flat:
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    print("=== Resume ===")
    for name, history in [("DDPG", ddpg_h), ("TD3", td3_h)]:
        evals = np.asarray(history.get("eval_mean_rewards", []), dtype=float)
        train = np.asarray(history.get("episode_rewards", []), dtype=float)
        best_eval = float(np.max(evals)) if len(evals) else float("nan")
        final_eval = float(evals[-1]) if len(evals) else float("nan")
        final_train = float(np.mean(train[-20:])) if len(train) else float("nan")
        print(f"{name} | best eval={best_eval:>8.1f} | final eval={final_eval:>8.1f} | "
              f"train avg20={final_train:>8.1f}")

**Lecture des six diagnostics.** La courbe d'évaluation déterministe est la métrique principale : les récompenses de collecte incluent le bruit d'exploration et ne mesurent pas directement la politique finale. Les valeurs $Q$ et l'erreur de Bellman indiquent ensuite si les critiques restent numériquement stables, tandis que l'écart $|Q_1-Q_2|$ révèle le désaccord propre à TD3. Enfin, les marques d'update acteur doivent être plus espacées pour TD3 : elles rendent visible le *policy delay*. Une loss plus basse ou un petit Q-gap ne suffit donc pas à conclure ; ces signaux doivent accompagner une amélioration de l'évaluation réelle.

In [ ]:
demo_results = {}

print("=== Démonstration DDPG (politique déterministe) ===")
demo_results["DDPG"] = evaluate_and_display_agent(agent, "DDPG")

print()
print("=== Démonstration TD3 (politique déterministe) ===")
demo_results["TD3"] = evaluate_and_display_agent(agent_td3, "TD3")


if demo_results:
    print()
    print("Résumé des démos déterministes :")
    for name, values in demo_results.items():
        print(f"  {name:4s} : {np.mean(values):+.1f} +/- {np.std(values):.1f}")


## 15. Conclusion et pont vers SAC

### Ce qu'on a construit

Deux algorithmes **actor-critic deterministes off-policy** sur HalfCheetah-v5 :

- **DDPG** : politique deterministe entrainee par le gradient d'un critique $Q(s,a)$, avec replay buffer, reseaux cibles et bruit d'exploration externe.
- **TD3** : la meme base, plus trois corrections qui reduisent les instabilites : twin critics, delayed policy updates, target policy smoothing.

Les briques fondamentales introduites :

| Brique | Role |
|--------|------|
| Theoreme DPG | Gradient deterministe = $\nabla_a Q \cdot \nabla_\theta \mu_\theta$ |
| `ContinuousReplayBuffer` | Buffer FIFO off-policy |
| Soft update | $\theta' \leftarrow \tau\theta + (1-\tau)\theta'$ |
| `TwinQNetwork` | Deux critiques + cible pessimiste $\min(Q_1, Q_2)$ |

### La progression des algorithmes off-policy

$$\underbrace{\text{DQN}}_{\text{discret}} \to \underbrace{\text{DDPG}}_{\text{continu det.}} \to \underbrace{\text{TD3}}_{\text{+ 3 corrections}} \to \underbrace{\text{SAC}}_{\text{stochastique + entropie}}$$

### Pont vers SAC

TD3 rend DDPG plus robuste, mais l'exploration reste externe. **SAC (Soft Actor-Critic)** change ce point : il revient a une politique stochastique off-policy et maximise une recompense augmentee par l'entropie :

$$\max_\pi \mathbb{E}\left[\sum_t r_t + \alpha \mathcal{H}[\pi(\cdot|s_t)]\right]$$

| | TD3 | SAC |
|-|-----|-----|
| Politique | deterministe | stochastique |
| Exploration | bruit externe | entropie apprise |
| Critics | twin Q | twin soft Q |
| Idee cle | limiter les updates trompeuses | apprendre performant et exploratoire |

SAC fera le pont entre la stabilite de TD3 et l'exploration naturelle des politiques stochastiques.

## References

- Silver, D., Lever, G., Heess, N., Degris, T., Wierstra, D. & Riedmiller, M. (2014). Deterministic Policy Gradient Algorithms. *ICML 2014*. [Proceedings](https://proceedings.mlr.press/v32/silver14.html). (Theoreme DPG : fondation theorique)

- Lillicrap, T. P., Hunt, J. J., Pritzel, A., Heess, N., Erez, T., Tassa, Y., Silver, D. & Wierstra, D. (2015). Continuous control with deep reinforcement learning. *ICLR 2016*. [arXiv:1509.02971](https://arxiv.org/abs/1509.02971). (DDPG)

- Fujimoto, S., van Hoof, H. & Meger, D. (2018). Addressing Function Approximation Error in Actor-Critic Methods. *ICML 2018*. [arXiv:1802.09477](https://arxiv.org/abs/1802.09477). (TD3 : twin critics, delayed updates, target smoothing)

- Haarnoja, T., Zhou, A., Abbeel, P. & Levine, S. (2018). Soft Actor-Critic. *ICML 2018*. [arXiv:1801.01290](https://arxiv.org/abs/1801.01290). (SAC : prochaine etape)

- Mnih, V. et al. (2015). Human-level control through deep reinforcement learning. *Nature 518*. (DQN : ancetre off-policy)

- Sutton, R. S. & Barto, A. G. (2018). *Reinforcement Learning: An Introduction* (2nd ed.). MIT Press. [En ligne](http://incompleteideas.net/book/the-book-2nd.html).

- OpenAI Spinning Up. *DDPG* et *TD3*. [spinningup.openai.com](https://spinningup.openai.com/en/latest/algorithms/ddpg.html).